# Baseline — TF-IDF

TF-IDF vektörleştirme ile temel ML sınıflandırıcılarının performansı.
BoW baseline ile karşılaştırmak için.

### Kütüphaneler

In [1]:
import pandas as pd
import numpy as np
import time
import copy
import warnings
import re
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb


### Veri Yükleme ve Bölme

In [2]:
df = pd.read_csv('spam_cleaned.csv', encoding='latin-1')
X, y = df['sms'], df['type']
print(f"Veri seti: {len(df)} satır | ham: {(y==0).sum()}, spam: {(y==1).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=111, stratify=y
)
print(f"Eğitim: {len(X_train)} | Test: {len(X_test)} | Train spam oranı: {y_train.mean()*100:.1f}%")


Veri seti: 5570 satır | ham: 4823, spam: 747
Eğitim: 3899 | Test: 1671 | Train spam oranı: 13.4%


### TF-IDF Vektörleştirme

Kelime sıklığını belge frekansının tersiyle ağırlıklandırır. Nadir ama ayırt edici kelimelere yüksek ağırlık verir.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)
print(f"TF-IDF özellik boyutu: {X_train_vec.shape[1]}")


TF-IDF özellik boyutu: 6911


### Değerlendirme Fonksiyonu

In [4]:
import scipy.stats as stats

def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred = clf.predict(X_te)
    try:
        y_score = clf.predict_proba(X_te)[:, 1]
    except AttributeError:
        y_score = clf.decision_function(X_te)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='accuracy')
    cv_acc = cv_scores.mean()
    cv_std = cv_scores.std()
    ci = stats.t.interval(0.95, df=len(cv_scores)-1, loc=cv_acc, scale=stats.sem(cv_scores))
    return {
        'Model':        name,
        'Accuracy':     round(accuracy_score(y_te, y_pred), 4),
        'Precision':    round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':       round(recall_score(y_te, y_pred), 4),
        'F1':           round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':      round(roc_auc_score(y_te, y_score), 4),
        'CV-Acc':       round(cv_acc, 4),
        'CV-Std':       round(cv_std, 4),
        'CI-95% Low':   round(ci[0], 4),
        'CI-95% High':  round(ci[1], 4),
        'Time(s)':      round(elapsed, 4),
    }

### Sınıflandırıcılar

In [5]:
classifiers = [
    ('LR',  LogisticRegression(max_iter=1000)),
    ('DT',  DecisionTreeClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=3)),
    ('SVM', SVC(kernel='linear', probability=True)),
    ('NC',  NearestCentroid()),
]

rows = []
for name, clf in classifiers:
    rows.append(evaluate(name, copy.deepcopy(clf), X_train_vec, y_train, X_test_vec, y_test))

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))


Model  Accuracy  Precision  Recall     F1  ROC-AUC  CV-Acc  CV-Std  CI-95% Low  CI-95% High  Time(s)
   LR    0.9707     0.9487  0.8259 0.8831   0.9887  0.9685  0.0025      0.9650       0.9719   0.0193
   DT    0.9731     0.8809  0.9241 0.9020   0.9524  0.9643  0.0068      0.9550       0.9737   0.0562
  KNN    0.9767     0.9426  0.8795 0.9099   0.9759  0.9779  0.0070      0.9682       0.9877   0.0003
  SVM    0.9826     0.9535  0.9152 0.9339   0.9903  0.9828  0.0033      0.9782       0.9874   1.1249
   NC    0.9533     0.8349  0.8125 0.8235   0.9950  0.9561  0.0065      0.9471       0.9652   0.0637
